# Notebook 05 — Final Composite Ranking & Figures

**Goal:** Integrate all scoring layers into a final composite ranking of
exercise mimetic candidates. Produce the three key figures for presentation.

**Composite score formula:**
```
composite = 0.4 * z(enrichment) + 0.4 * z(gsfm_similarity) + 0.2 * z(anti_aging)
            + bonus(IDG_novelty)
```
Equal-weighted and enrichment-dominant weightings are both reported.

**Three key figures:**
1. Workflow diagram placeholder (build in draw.io / Biorender externally)
2. Heatmap — top 30 compounds × tissues, connectivity score
3. UMAP — GSFM embedding space with exercise signatures

**Outputs:**
- `results/final_ranked_compounds.csv` — complete annotated ranking
- `results/figures/05_heatmap_top30.png`
- `results/figures/05_composite_score_comparison.png`

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import signatures as sig
from src import connectivity as conn

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

TISSUES       = list(sig.TISSUES.keys())
TIMEPOINT     = '8w'
TOP_N_FINAL   = 30     # compounds in heatmap
IDG_BONUS     = 0.5    # z-score bonus for Tbio/Tdark hits

# Weighting schemes to compare
WEIGHT_SCHEMES = {
    'equal':       {'enrichment': 0.4, 'gsfm': 0.4, 'aging': 0.2},
    'enrich_dom':  {'enrichment': 0.6, 'gsfm': 0.2, 'aging': 0.2},
    'gsfm_dom':    {'enrichment': 0.2, 'gsfm': 0.6, 'aging': 0.2},
}

PROCESSED_DIR = Path('../data/processed')
RESULTS_DIR   = Path('../results')
FIGURES_DIR   = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration ready.')

In [ ]:
# ── Load all scoring inputs ───────────────────────────────────────────────────

def load_parquet_safe(path: Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_parquet(path)
        print(f'Loaded {label}: {len(df)} rows')
        return df
    else:
        print(f'WARNING: {label} not found at {path}')
        return pd.DataFrame()

ranked_meta  = load_parquet_safe(PROCESSED_DIR / 'ranked_compounds_meta.parquet',
                                  'meta ranking')
annotated    = load_parquet_safe(PROCESSED_DIR / 'annotated_compounds.parquet',
                                  'annotated compounds')

# Per-tissue GSFM scores
gsfm_scores = {}
for tissue in TISSUES:
    df = load_parquet_safe(PROCESSED_DIR / f'gsfm_ranked_{tissue}.parquet',
                            f'GSFM {tissue}')
    if not df.empty:
        gsfm_scores[tissue] = df.set_index('pert_iname')['gsfm_cosine']

# Per-tissue enrichment scores
enrich_scores = {}
for tissue in TISSUES:
    df = load_parquet_safe(PROCESSED_DIR / f'lincs_ranked_{tissue}.parquet',
                            f'enrichment {tissue}')
    if not df.empty:
        score_col = 'median_score' if 'median_score' in df.columns else df.columns[1]
        enrich_scores[tissue] = df.set_index('pert_iname')[score_col]

print('\nAll inputs loaded.')

In [ ]:
# ── Build composite scoring table ─────────────────────────────────────────────

# Use annotated if available, fall back to ranked_meta
base = annotated if not annotated.empty else ranked_meta
if base.empty:
    raise RuntimeError('No ranking data available. Run notebooks 02 and 04 first.')

master = base.copy()

def zscore_col(series: pd.Series) -> pd.Series:
    """Robust z-score handling NaN and constant series."""
    s = series.fillna(series.median())
    std = s.std()
    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)
    return (s - s.mean()) / std

# Z-score each scoring dimension
enrich_col = 'mean_score' if 'mean_score' in master.columns else master.columns[1]
master['z_enrichment'] = zscore_col(master[enrich_col])

# Mean GSFM cosine across tissues
if gsfm_scores:
    gsfm_mean = pd.concat(gsfm_scores.values(), axis=1).mean(axis=1)
    master['gsfm_mean_cosine'] = master['pert_iname'].map(gsfm_mean)
    master['z_gsfm'] = zscore_col(master['gsfm_mean_cosine'])
else:
    master['z_gsfm'] = 0.0
    print('GSFM scores not available — setting z_gsfm = 0')

# Aging concordance
if 'anti_aging_score' in master.columns:
    master['z_aging'] = zscore_col(master['anti_aging_score'])
else:
    master['z_aging'] = 0.0
    print('Aging concordance not available — setting z_aging = 0')

# IDG novelty bonus
if 'has_novel_target' in master.columns:
    master['idg_bonus'] = master['has_novel_target'].fillna(False).astype(float) * IDG_BONUS
else:
    master['idg_bonus'] = 0.0

# Compute composite scores for each weight scheme
for scheme_name, weights in WEIGHT_SCHEMES.items():
    master[f'composite_{scheme_name}'] = (
        weights['enrichment'] * master['z_enrichment'] +
        weights['gsfm']       * master['z_gsfm']       +
        weights['aging']      * master['z_aging']       +
        master['idg_bonus']
    )

# Primary composite
master['composite_score'] = master['composite_equal']
master = master.sort_values('composite_score', ascending=False).reset_index(drop=True)
master['final_rank'] = range(1, len(master) + 1)

print(f'Composite scoring complete for {len(master)} compounds.')
print(f'\nTop 20 final-ranked compounds:')
display_cols = ['final_rank', 'pert_iname', 'composite_score', 'z_enrichment',
                'z_gsfm', 'z_aging', 'idg_bonus', 'n_tissues']
display_cols = [c for c in display_cols if c in master.columns]
print(master.head(20)[display_cols].to_string(index=False))

In [ ]:
# ── Robustness check — rank correlation across weight schemes ─────────────────

from scipy.stats import spearmanr

scheme_names = list(WEIGHT_SCHEMES.keys())
score_matrix = master[[f'composite_{s}' for s in scheme_names]].copy()
score_matrix.columns = scheme_names

corr_matrix = pd.DataFrame(index=scheme_names, columns=scheme_names, dtype=float)
for a in scheme_names:
    for b in scheme_names:
        r, p = spearmanr(score_matrix[a], score_matrix[b])
        corr_matrix.loc[a, b] = r

print('Rank correlation across weighting schemes (Spearman rho):')
print(corr_matrix.round(3))
print()
print('High correlation (>0.9) indicates top hits are robust to weight choice.')

In [ ]:
# ── Figure 2: Heatmap — top 30 compounds × tissues ───────────────────────────
# Shows enrichment connectivity score (normalized) for the top-30 meta-ranked
# compounds, annotated with mechanism class on the y-axis.

top30 = master.head(TOP_N_FINAL)['pert_iname'].tolist()

# Build matrix: compounds × tissues
heatmap_data = {}
for tissue in TISSUES:
    if tissue in enrich_scores:
        s = enrich_scores[tissue]
        heatmap_data[sig.TISSUES[tissue]['display']] = [
            s.get(c, np.nan) for c in top30
        ]

if heatmap_data:
    hm_df = pd.DataFrame(heatmap_data, index=top30)

    # Row annotations: MOA and IDG TDL
    row_colors_moa  = []
    row_colors_tdl  = []

    moa_palette = sns.color_palette('tab20', n_colors=20)
    tdl_palette = {
        'Tclin': '#2196F3', 'Tchem': '#4CAF50',
        'Tbio':  '#FF9800', 'Tdark': '#9C27B0', None: '#CCCCCC'
    }

    moa_vals = []
    tdl_vals  = []
    for compound in top30:
        row = master[master['pert_iname'] == compound]
        moa_vals.append(row['moa'].iloc[0] if not row.empty and 'moa' in row.columns else '')
        tdl_vals.append(row['best_tdl_label'].iloc[0]
                        if not row.empty and 'best_tdl_label' in row.columns else None)

    unique_moa = list(dict.fromkeys([m for m in moa_vals if m]))
    moa_color_map = {m: moa_palette[i % 20] for i, m in enumerate(unique_moa)}
    row_moa_colors = pd.Series(
        [moa_color_map.get(m, (0.8, 0.8, 0.8)) for m in moa_vals], index=top30
    )
    row_tdl_colors = pd.Series(
        [tdl_palette.get(t, '#CCCCCC') for t in tdl_vals], index=top30
    )

    # Clustermap
    row_colors = pd.DataFrame({'MOA': row_moa_colors, 'TDL': row_tdl_colors})

    g = sns.clustermap(
        hm_df.fillna(0),
        cmap='RdBu_r', center=0,
        row_colors=row_colors,
        col_cluster=False,
        row_cluster=True,
        figsize=(max(6, len(heatmap_data) * 2 + 3), max(10, len(top30) * 0.4)),
        xticklabels=True, yticklabels=True,
        dendrogram_ratio=(0.15, 0.05),
        cbar_pos=(1.02, 0.3, 0.03, 0.4),
    )
    g.ax_heatmap.set_title(
        f'Connectivity score: top-{TOP_N_FINAL} exercise mimetics × tissues',
        pad=20, fontsize=12
    )
    g.ax_heatmap.set_yticklabels(
        g.ax_heatmap.get_yticklabels(), fontsize=8
    )

    # TDL legend
    tdl_patches = [mpatches.Patch(color=c, label=t) for t, c in tdl_palette.items() if t]
    g.ax_heatmap.legend(
        handles=tdl_patches, title='IDG TDL',
        loc='upper left', bbox_to_anchor=(1.25, 1.05), fontsize=8
    )

    plt.savefig(
        FIGURES_DIR / '05_heatmap_top30.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()
    print(f'Saved heatmap: {FIGURES_DIR / "05_heatmap_top30.png"}')
else:
    print('Heatmap skipped: per-tissue enrichment scores not available.')

In [ ]:
# ── Figure 3: Composite score comparison across weighting schemes ─────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

top50 = master.head(50)

for ax, scheme_name in zip(axes, scheme_names):
    sorted_df = top50.sort_values(f'composite_{scheme_name}', ascending=True)
    colors = ['firebrick' if c in conn.KNOWN_MIMETICS else 'steelblue'
              for c in sorted_df['pert_iname'].str.lower()]
    ax.barh(
        range(len(sorted_df)),
        sorted_df[f'composite_{scheme_name}'],
        color=colors, edgecolor='none', alpha=0.85
    )
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df['pert_iname'], fontsize=6)
    ax.set_xlabel('Composite z-score')
    weights = WEIGHT_SCHEMES[scheme_name]
    ax.set_title(
        f'{scheme_name}\n'
        f'enrich={weights["enrichment"]}, gsfm={weights["gsfm"]}, aging={weights["aging"]}',
        fontsize=9
    )

    # Red = known mimetics
    known_patch = mpatches.Patch(color='firebrick', label='Known mimetic')
    novel_patch = mpatches.Patch(color='steelblue', label='Novel candidate')
    ax.legend(handles=[known_patch, novel_patch], fontsize=7, loc='lower right')

plt.suptitle('Top-50 exercise mimetic candidates — composite score sensitivity',
             fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_composite_score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved composite score comparison figure.')

In [ ]:
# ── Save final ranked compound table ─────────────────────────────────────────

output_cols = [
    'final_rank', 'pert_iname', 'composite_score',
    'composite_equal', 'composite_enrich_dom', 'composite_gsfm_dom',
    'z_enrichment', 'z_gsfm', 'z_aging',
    'mean_score', 'n_tissues', 'tissues_ranked',
    'moa', 'target',
    'best_tdl_label', 'has_novel_target', 'novel_targets',
    'anti_aging_score',
    'idg_bonus',
]
output_cols = [c for c in output_cols if c in master.columns]
final_df = master[output_cols].copy()

out_csv = RESULTS_DIR / 'final_ranked_compounds.csv'
final_df.to_csv(out_csv, index=False)
print(f'Saved final ranking: {out_csv}')
print(f'  {len(final_df)} compounds × {len(final_df.columns)} columns')

In [ ]:
# ── Top-line summary for the talk ─────────────────────────────────────────────

n_total      = len(master)
threshold    = master['composite_score'].quantile(0.95)
n_above_thr  = (master['composite_score'] >= threshold).sum()

known_top    = master.head(50)['pert_iname'].str.lower()
known_found  = [m for m in conn.KNOWN_MIMETICS if m.lower() in set(known_top)]

novel_top    = master.head(50)
novel_count  = novel_top['has_novel_target'].sum() if 'has_novel_target' in novel_top.columns else 'N/A'

print('=' * 60)
print('TOP-LINE SUMMARY')
print('=' * 60)
print(f'Total LINCS compounds evaluated:    {n_total:,}')
print(f'Compounds with composite score >    ')
print(f'  95th percentile ({threshold:.2f}):      {n_above_thr}')
print()
print(f'Known exercise mimetics in top-50:  {known_found}')
print(f'Novel-target candidates in top-50:  {novel_count}')
print()
print('Top 10 final candidates:')
print('-' * 60)
for i, row in master.head(10).iterrows():
    label = '(known)' if row.get('pert_iname', '').lower() in conn.KNOWN_MIMETICS else ''
    novel = '[NOVEL TARGET]' if row.get('has_novel_target', False) else ''
    print(f"  {row.get('final_rank', i+1):2d}. {row['pert_iname']:<25s} "
          f"composite={row['composite_score']:5.2f} {label} {novel}")

print()
print('Key caveats:')
print('  - Cell lines used: HA1E, A549, MCF7, PC3 (not rat muscle/liver/adipose)')
print('  - L1000 covers 978 landmark genes only')
print('  - Rat→human ortholog mapping loses ~10-15% of genes')
print('  - GSFM embeddings: compound signatures approximated from overlap genes')
print()
print('=== Pipeline complete ===')
print(f'Final table: results/final_ranked_compounds.csv')
print(f'Figures:     results/figures/')
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f'  {f.name}')